# 🔌 Model Context Protocol (MCP) — The USB-C Port for AI

## From Manual Tool Plumbing to Plug-and-Play Intelligence

---

**What you'll learn in this notebook:**

1. Why connecting LLMs to the real world is painful today  
2. What MCP is, and the architecture behind it  
3. Key vocabulary — Servers, Clients, Hosts, Tools, Resources, Transports  
4. A hands-on "before & after" comparison — building the same integration *without* MCP, then *with* MCP  
5. How to build your own MCP Server from scratch  
6. How to build an MCP Client and wire it to OpenAI  
7. Using the OpenAI Agents SDK with MCP for real-world workflows  

**Prerequisites:** Basic Python, familiarity with calling APIs, a curiosity about how AI agents talk to the outside world.

**Time:** ~2.5 hours

---

## 0 · Environment Setup

Let's install everything we need up front. We'll use:
- **`openai`** — to call GPT models via the **Responses API** (the current standard)  
- **`mcp`** — the official MCP Python SDK (includes FastMCP)  
- **`openai-agents`** — OpenAI's Agents SDK with built-in MCP support  
- **`httpx`** — async HTTP client  
- **`nest_asyncio`** — to allow async code in Jupyter  

In [26]:
# ── Install Dependencies ──────────────────────────────────────
!pip install openai mcp httpx openai-agents nest_asyncio pydantic -q


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [27]:
# ── Imports & Async Patch ─────────────────────────────────────
import nest_asyncio
nest_asyncio.apply()   # Lets us use asyncio.run() inside Jupyter

import os
import json
import asyncio
import httpx
from openai import OpenAI
from pprint import pprint

In [28]:
# !pip install openai python-dotenv pandas
import pandas as pd
import os, json, time
from dotenv import load_dotenv
from openai import OpenAI
import textwrap

import truststore
truststore.inject_into_ssl()



def pretty_print(*args):
    text = " ".join(str(arg) for arg in args)
    try:
        print(textwrap.fill(text, width=80))
    except Exception as e:
        print(text)  # fallback to normal print if text is not a string

        

load_dotenv('/Users/shivam13juna/Documents/scaler/iitr_classes/llm_ref/openai_key.env')  # reads .env file in the current directory

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError(
        "OPENAI_API_KEY not found! "
        "Make sure you have a .env file with: OPENAI_API_KEY=sk-..."
    )

pretty_print("API key loaded successfully.")

MODEL = 'gpt-5-nano'




client = OpenAI(api_key=api_key)
pretty_print("OpenAI client ready.")

API key loaded successfully.
OpenAI client ready.


---

# Part 1 · The Problem — Why Do LLMs Need an "Integration Standard"?

Imagine you're building a helpful AI assistant for your company. It needs to:

- 📧 Read and send emails (Gmail)  
- 📅 Check and create calendar events (Google Calendar)  
- 📁 Search company documents (Google Drive)  
- 🌤️ Check the weather before suggesting outdoor meetings  
- 🐛 Create bug tickets (Jira / GitHub Issues)  

**The core challenge:** LLMs like GPT-4o are incredibly smart *thinkers*, but they live in a text-in-text-out box. They can't natively browse the web, call APIs, or read your files. To make them *useful*, you need to connect them to external tools.

### The Naive Approach — "I'll just write some glue code"

For each tool, you'd need to:
1. Write the function that calls the external API  
2. Define the tool schema so the LLM knows how to call it  
3. Parse the LLM's function-call response  
4. Actually execute the function  
5. Feed the result back to the LLM  
6. Handle errors, retries, auth tokens…  

Let's see what this looks like in practice.

## 1.1 · The World Without MCP — Manual Tool Wiring

Let's connect GPT-4o-mini to a simple weather API **by hand** using OpenAI's **Responses API**. We'll use the free [Open-Meteo API](https://open-meteo.com/) — no API key required.

Pay close attention to how much boilerplate we need to write just for **one** tool.

In [29]:
# ══════════════════════════════════════════════════════════════
# STEP 1: Write the actual function that calls the external API
# ══════════════════════════════════════════════════════════════

def get_weather_manual(latitude: float, longitude: float) -> dict:
    """Call the Open-Meteo API to get current weather for given coordinates."""
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "current_weather": True
    }
    response = httpx.get(url, params=params)
    response.raise_for_status()
    data = response.json()
    weather = data["current_weather"]
    return {
        "temperature_celsius": weather["temperature"],
        "windspeed_kmh": weather["windspeed"],
        "weather_code": weather["weathercode"],
        "is_day": weather["is_day"] == 1
    }


import httpx

#def geocode_city(city: str):
#    url = "https://nominatim.openstreetmap.org/search"
#    params = {"q": city, "format": "json", "limit": 1}
#    headers = {"User-Agent": "mcp-workshop-demo/1.0"}
#    r = httpx.get(url, params=params, headers=headers, timeout=20)
#    r.raise_for_status()
#    data = r.json()
#    if not data:
#        return None
#    return {"latitude": float(data[0]["lat"]), "longitude": float(data[0]["lon"])}

#print(geocode_city("Tokyo"))

# Quick test
result = get_weather_manual(48.8566, 2.3522)  # Paris
print("Direct API call result for Paris:")
pprint(result)

Direct API call result for Paris:
{'is_day': False,
 'temperature_celsius': 7.4,
 'weather_code': 2,
 'windspeed_kmh': 4.7}


In [30]:
# ══════════════════════════════════════════════════════════════
# STEP 2: Define the Tool Schema for the Responses API
#
# Note: The Responses API uses a FLAT tool format:
#   - "type": "function"
#   - "name": directly at top level (not nested under "function")
#   - "parameters": directly at top level
# ══════════════════════════════════════════════════════════════

weather_tool_schema = {
    "type": "function",
    "name": "get_weather",
    "description": "Get the current weather for a location given its latitude and longitude. "
                   "Returns temperature in Celsius, wind speed, and weather conditions.",
    "parameters": {
        "type": "object",
        "properties": {
            "latitude": {
                "type": "number",
                "description": "Latitude of the location (e.g., 48.8566 for Paris)"
            },
            "longitude": {
                "type": "number",
                "description": "Longitude of the location (e.g., 2.3522 for Paris)"
            }
        },
        "required": ["latitude", "longitude"],
        "additionalProperties": False
    },
    "strict": True
}

print("Tool schema defined ✅")
print(json.dumps(weather_tool_schema, indent=2))

Tool schema defined ✅
{
  "type": "function",
  "name": "get_weather",
  "description": "Get the current weather for a location given its latitude and longitude. Returns temperature in Celsius, wind speed, and weather conditions.",
  "parameters": {
    "type": "object",
    "properties": {
      "latitude": {
        "type": "number",
        "description": "Latitude of the location (e.g., 48.8566 for Paris)"
      },
      "longitude": {
        "type": "number",
        "description": "Longitude of the location (e.g., 2.3522 for Paris)"
      }
    },
    "required": [
      "latitude",
      "longitude"
    ],
    "additionalProperties": false
  },
  "strict": true
}


In [31]:
# ══════════════════════════════════════════════════════════════
# STEP 3: Build the full tool-use loop using the RESPONSES API
#
# Key differences from the older Chat Completions API:
#  - client.responses.create() instead of client.chat.completions.create()
#  - "input" instead of "messages"
#  - Tool calls appear as function_call items in response.output
#  - Tool results are sent back as function_call_output items
# ══════════════════════════════════════════════════════════════

def chat_with_weather_tool(user_message: str) -> str:
    """
    A complete manual tool-use loop with the Responses API:
    1. Send user message + tool definitions
    2. Check if the model wants to call a function
    3. Execute the function locally
    4. Send the result back
    5. Return the final answer
    """

    # ── First call: the model may request a tool call ──
    response = client.responses.create(
        model=MODEL,
        instructions="You are a helpful assistant. Use the get_weather tool when asked about weather.",
        input=user_message,
        tools=[weather_tool_schema],
    )

    while True:
        function_calls = [item for item in response.output if item.type == "function_call"]

        if not function_calls:
            return response.output_text

        tool_outputs = []

        for fc in function_calls:
            arguments = json.loads(fc.arguments)

            print(f"🔧 LLM requested tool: {fc.name}")
            print(f"   Arguments: {arguments}")

            if fc.name == "get_weather":
                tool_result = get_weather_manual(
                    latitude=arguments["latitude"],
                    longitude=arguments["longitude"],
                )
            else:
                tool_result = {"error": f"Unknown tool: {fc.name}"}

            print(f"   Result: {tool_result}")

            tool_outputs.append({
                "type": "function_call_output",
                "call_id": fc.call_id,
                "output": json.dumps(tool_result),
            })

        response = client.responses.create(
            model=MODEL,
            instructions="You are a deadpool, but have been hired as assistant.",
            previous_response_id=response.id,   # key fix
            input=tool_outputs,                 # only tool outputs
            tools=[weather_tool_schema],        # keep tools available for another round
        )
        return response.output_text

    else:
        return response.output_text

# Let's test it!
answer = chat_with_weather_tool("What's the weather like in Tokyo right now?")
print("\n" + "="*60)
print("💬 Final Answer:")
print(answer)

🔧 LLM requested tool: get_weather
   Arguments: {'latitude': 35.6762, 'longitude': 139.6503}
   Result: {'temperature_celsius': 12.7, 'windspeed_kmh': 5.7, 'weather_code': 1, 'is_day': True}

💬 Final Answer:
Tokyo right now: about 12.7°C with a light breeze around 5.7 km/h. It’s daytime. Want me to get a more detailed forecast or a packing tip for the day?


### 🤔 Reflection: What Did We Just Build?

Let's count the pieces of **manual plumbing** we wrote for just **ONE tool**:

| # | What We Had To Write | Lines |
|---|---------------------|-------|
| 1 | The function itself (`get_weather_manual`) | ~15 |
| 2 | The tool schema (tool definition) | ~25 |
| 3 | The conversation loop with tool routing | ~50 |
| 4 | Argument parsing (`json.loads`) | scattered |
| 5 | Error handling (barely any!) | 0 |
| **Total** | **~90 lines for ONE tool** | |

Now imagine doing this for **10 tools**. Or **50**. Each tool needs:
- Its own function  
- Its own JSON schema (kept in sync with the function — good luck!)  
- A new `elif` branch in the routing logic  
- Its own error handling  
- Its own authentication management  

**This is the N × M integration problem.** Every new tool × every new LLM application = custom glue code. It doesn't scale.

> 🔑 **Key Insight:** We need a *standard protocol* — like USB or HTTP — that lets any tool talk to any LLM application, with zero custom glue code.

**That's exactly what MCP is.**

---

# Part 2 · Enter MCP — The Model Context Protocol

## 2.1 What is MCP?

**MCP (Model Context Protocol)** is an open protocol — created by Anthropic and now hosted by the Linux Foundation — that standardizes how LLM applications connect to external data sources and tools.

Think of it like **USB-C for AI:**

| Analogy | Without Standard | With Standard |
|---------|-----------------|---------------|
| **Hardware** | Every device had its own charger/port | USB-C: one cable for everything |
| **Web** | Every browser spoke differently to servers | HTTP: one protocol for the web |
| **AI Tools** | Every tool needs custom glue code | MCP: one protocol for all tools |

## 2.2 The Architecture

MCP follows a **client-server architecture** with three main roles:

```
┌─────────────────────────────────────────────────────────────────────┐
│                         HOST APPLICATION                            │
│                    (Claude Desktop, IDE, Your App)                   │
│                                                                     │
│  ┌─────────────┐  ┌─────────────┐  ┌─────────────┐                │
│  │ MCP Client  │  │ MCP Client  │  │ MCP Client  │                │
│  │     #1      │  │     #2      │  │     #3      │                │
│  └──────┬──────┘  └──────┬──────┘  └──────┬──────┘                │
│         │                │                │                         │
└─────────┼────────────────┼────────────────┼─────────────────────────┘
          │                │                │
          ▼                ▼                ▼
   ┌──────────────┐ ┌──────────────┐ ┌──────────────┐
   │  MCP Server  │ │  MCP Server  │ │  MCP Server  │
   │   Weather    │ │  Database    │ │   GitHub     │
   │    API       │ │  Access      │ │   Issues     │
   └──────────────┘ └──────────────┘ └──────────────┘
```

### Glossary

| Term | What It Is | Real-World Example |
|------|-----------|-------------------|
| **Host** | The application the user interacts with. It manages one or more MCP clients. | Claude Desktop, VS Code + Copilot, your custom chatbot |
| **MCP Client** | Lives *inside* the host. Maintains a 1:1 session with a single MCP server. | A client object your app creates per server connection |
| **MCP Server** | A lightweight program that exposes tools, resources, and prompts via the MCP protocol. | A weather server, a GitHub server, a database server |
| **Tool** | A function the LLM can *call* (think POST endpoint). Has a name, description, and input schema. | `get_weather(lat, lon)`, `create_issue(title, body)` |
| **Resource** | Data the LLM can *read* (think GET endpoint). Identified by a URI. | `file://docs/readme.md`, `db://users/123` |
| **Prompt** | A reusable template for LLM interactions, exposed by the server. | "Summarize this document", "Review this code for security issues" |
| **Transport** | *How* client and server communicate. | **stdio** (local subprocess), **Streamable HTTP** (remote), **SSE** (legacy remote) |


## 2.3 How MCP Communication Works

There are really **two phases**: **tool discovery** and **tool execution**.

### Phase 1: Tool Discovery

Before the LLM can decide to call something like `get_weather`, the host application must first learn what tools the MCP server provides.

```
Host application starts / connects to MCP server
           │
           ▼
┌─────────────────────┐
│   HOST APPLICATION   │
│                      │
│  ┌────────────────┐  │  1. Host initializes MCP connection
│  │   MCP Client   │──┼──── 2. Client requests available tools ────────┐
│  └────────────────┘  │                                                 │
│                      │                                                 ▼
│                      │                                    ┌────────────────────┐
│                      │                                    │    MCP Server      │
│                      │                                    │   (Weather)        │
│                      │                                    │                    │
│                      │◄──── 3. Server returns tool list ──│  Tools:            │
│                      │                                    │  - get_weather     │
│                      │                                    │  - get_forecast    │
│                      │                                    └────────────────────┘
│                      │
│  ┌────────────────┐  │  4. Host converts MCP tool schemas
│  │   LLM Engine   │◄─┼──── into the LLM's tool/function format
│  └────────────────┘  │
└─────────────────────┘
```

At this point, the LLM knows that a tool named `get_weather` exists, what it does, and what arguments it expects.

### Phase 2: Tool Execution

Now when the user asks a question, the LLM can choose one of the discovered tools.

```
User: "What's the weather in Tokyo?"
           │
           ▼
┌─────────────────────┐
│   HOST APPLICATION   │  1. User sends message
│                      │  2. Host forwards message + available tool definitions to LLM
│  ┌────────────────┐  │  3. LLM says: "Call get_weather(location='Tokyo')"
│  │   LLM Engine   │  │
│  └───────┬────────┘  │
│          │           │  4. Host tells MCP Client to invoke that tool
│  ┌───────▼────────┐  │
│  │   MCP Client   │──┼──── 5. Client sends tool request to server ────┐
│  └───────▲────────┘  │                                                 │
│          │           │                                                 ▼
│          │           │                                    ┌────────────────────┐
│          │           │                                    │    MCP Server      │
│          │           │                                    │   (Weather)        │
│          │           │                                    │                    │
│          │           │  6. Server calls weather API       │                    │
│          │           │                                    └────────────────────┘
│  ┌───────┴────────┐  │
│  │   LLM Engine   │◄─┼──── 7. Client returns tool result to host
│  │  (with result) │  │      and host passes it back to LLM
│  └───────┬────────┘  │
│          │           │  8. LLM generates natural-language answer
└──────────┼───────────┘
           ▼
User: "It's 22°C and sunny in Tokyo right now!"
```

The key insight: **the LLM never talks directly to the MCP server.** The host application orchestrates everything. The MCP client handles protocol details like discovery, serialization, transport, and error handling. The LLM only sees the tool definitions the host gives it, and later the tool results the host sends back.

## 2.4 What Does MCP Give Us?

| Without MCP                                                          | With MCP                                                           |
| -------------------------------------------------------------------- | ------------------------------------------------------------------ |
| Custom JSON schema per tool, per app                                 | Server defines tools once; clients can discover them automatically |
| Manual function routing in each app                                  | Protocol standardizes discovery and invocation                     |
| Tool definitions hardcoded into hosts                                | Host can fetch tools dynamically from servers                      |
| Auth logic scattered everywhere                                      | Server manages its own auth                                        |
| Schema drift between implementation and exposed function definitions | Tool schemas come directly from the server's definitions           |
| N tools × M apps = N×M integrations                                  | N tools + M apps = N+M integrations                                |

## Important Clarification

What actually happens is:

1. The **host** uses the **MCP client** to ask the server for its tools.
2. The **server** returns tool names, descriptions, and schemas.
3. The **host** passes those tool definitions to the **LLM**.
4. The **LLM** decides which tool to call based on that information.

---

# Part 3 · Building Your Own MCP Server from Scratch

Now let's build the **exact same weather tool** — but as a proper MCP Server using the official Python SDK.

You'll see how much cleaner this is.

## 3.1 · The FastMCP Framework

The official Python SDK provides **FastMCP** — a high-level framework that uses decorators and type hints to auto-generate tool schemas. No more hand-writing JSON schemas!

Let's create our weather MCP server in a Python file:

In [32]:
# ══════════════════════════════════════════════════════════════
# Let's write our MCP Server to a file
# We'll run it as a subprocess later
# ══════════════════════════════════════════════════════════════

weather_server_code = '''
import json
import httpx
from mcp.server.fastmcp import FastMCP

# ── Initialize the MCP Server ────────────────────────────────
mcp = FastMCP("weather_mcp")


@mcp.tool()
async def get_weather(latitude: float, longitude: float) -> str:
    """Get the current weather for a location.

    Args:
        latitude: Latitude of the location (e.g., 48.8566 for Paris)
        longitude: Longitude of the location (e.g., 2.3522 for Paris)

    Returns:
        A JSON string with temperature, wind speed, and conditions.
    """
    url = "https://api.open-meteo.com/v1/forecast"
    params = {"latitude": latitude, "longitude": longitude, "current_weather": True}

    async with httpx.AsyncClient() as http_client:
        response = await http_client.get(url, params=params)
        response.raise_for_status()
        data = response.json()

    weather = data["current_weather"]
    return json.dumps({
        "temperature_celsius": weather["temperature"],
        "windspeed_kmh": weather["windspeed"],
        "weather_code": weather["weathercode"],
        "is_day": weather["is_day"] == 1
    }, indent=2)


@mcp.tool()
async def get_weather_comparison(
    city1_lat: float, city1_lon: float, city1_name: str,
    city2_lat: float, city2_lon: float, city2_name: str
) -> str:
    """Compare current weather between two cities.

    Args:
        city1_lat: Latitude of the first city
        city1_lon: Longitude of the first city
        city1_name: Name of the first city
        city2_lat: Latitude of the second city
        city2_lon: Longitude of the second city
        city2_name: Name of the second city
    """
    async with httpx.AsyncClient() as http_client:
        cities = {}
        for name, lat, lon in [(city1_name, city1_lat, city1_lon),
                                (city2_name, city2_lat, city2_lon)]:
            resp = await http_client.get(
                "https://api.open-meteo.com/v1/forecast",
                params={"latitude": lat, "longitude": lon, "current_weather": True}
            )
            resp.raise_for_status()
            w = resp.json()["current_weather"]
            cities[name] = {"temperature_celsius": w["temperature"], "windspeed_kmh": w["windspeed"]}

    return json.dumps({
        "comparison": cities,
        "warmer_city": max(cities, key=lambda c: cities[c]["temperature_celsius"]),
        "windier_city": max(cities, key=lambda c: cities[c]["windspeed_kmh"]),
    }, indent=2)


# ── Define a Resource (read-only data endpoint) ─────────────
WEATHER_CODES = {
    0: "Clear sky", 1: "Mainly clear", 2: "Partly cloudy", 3: "Overcast",
    45: "Fog", 48: "Depositing rime fog",
    51: "Light drizzle", 53: "Moderate drizzle", 55: "Dense drizzle",
    61: "Slight rain", 63: "Moderate rain", 65: "Heavy rain",
    71: "Slight snowfall", 73: "Moderate snowfall", 75: "Heavy snowfall",
    95: "Thunderstorm", 96: "Thunderstorm with slight hail",
    99: "Thunderstorm with heavy hail",
}

@mcp.resource("weather://codes")
def get_weather_codes() -> str:
    """A lookup table of WMO weather interpretation codes."""
    return json.dumps(WEATHER_CODES, indent=2)


if __name__ == "__main__":
    mcp.run()  # Uses stdio transport by default
'''

with open("weather_server.py", "w") as f:
    f.write(weather_server_code)

print("✅ weather_server.py written!")
print()
print("Let's compare what FastMCP does for us vs the manual approach:")
print()
print("MANUAL APPROACH               │  MCP SERVER APPROACH")
print("──────────────────────────────│──────────────────────────")
print("Hand-write JSON schema (25 ln)│  @mcp.tool() decorator (0 ln!)")
print("Build routing logic (if/elif) │  Protocol handles routing")
print("Manage function registry      │  Server auto-registers tools")
print("Parse arguments manually      │  Type hints → auto-validation")
print("No discoverability            │  Client can list_tools()")
print("Auth scattered in app code    │  Server owns its own auth")

✅ weather_server.py written!

Let's compare what FastMCP does for us vs the manual approach:

MANUAL APPROACH               │  MCP SERVER APPROACH
──────────────────────────────│──────────────────────────
Hand-write JSON schema (25 ln)│  @mcp.tool() decorator (0 ln!)
Build routing logic (if/elif) │  Protocol handles routing
Manage function registry      │  Server auto-registers tools
Parse arguments manually      │  Type hints → auto-validation
No discoverability            │  Client can list_tools()
Auth scattered in app code    │  Server owns its own auth


### 🔍 Key Observations

**Look how little code we needed!** The `@mcp.tool()` decorator does all the heavy lifting:
- **Schema generation**: It reads the function signature (`latitude: float, longitude: float`) and docstring to auto-generate the exact same JSON schema we wrote by hand earlier.
- **Input validation**: Type hints are enforced automatically.
- **Registration**: The tool is registered with the server — no manual routing needed.

We also added:
- A **second tool** (`get_weather_comparison`) — adding new tools is just adding new decorated functions!
- A **resource** (`weather://codes`) — a read-only data endpoint the LLM can query to understand weather codes.

Let's verify the server is syntactically valid:

In [33]:
# Verify our server file is valid Python
import py_compile
try:
    py_compile.compile("weather_server.py", doraise=True)
    print("✅ weather_server.py compiles successfully!")
except py_compile.PyCompileError as e:
    print(f"❌ Compilation error: {e}")

✅ weather_server.py compiles successfully!


---

# Part 4 · Building an MCP Client — Talking to Our Server

Now let's build the **other side of the equation** — an MCP Client that:
1. Connects to our weather server  
2. Discovers what tools it offers  
3. Calls those tools  

This demonstrates the key MCP superpower: **the client doesn't need to know anything about the server's internals ahead of time.**

In [34]:
# ══════════════════════════════════════════════════════════════
# MCP Client — Connect, Discover, and Call Tools
# ══════════════════════════════════════════════════════════════

# ClientSession: high-level MCP session wrapper over transport streams.
# StdioServerParameters: tells MCP how to launch the local server process.
from mcp import ClientSession, StdioServerParameters

# stdio_client: opens stdin/stdout pipes to the server subprocess.
from mcp.client.stdio import stdio_client


async def demo_mcp_client():
    """
    Demonstrates the MCP client lifecycle:
    1) Start/attach to MCP server over stdio
    2) Initialize protocol handshake
    3) Discover tools and resources
    4) Invoke a tool with typed arguments
    5) Read a resource by URI
    """

    # Configure how to start the MCP server process.
    # Equivalent shell command: python weather_server.py
    server_params = StdioServerParameters(
        command="python",
        args=["weather_server.py"],
    )

    print("🔌 Connecting to weather_server.py via stdio...")
    print()

    # Open bidirectional streams to the MCP server process.
    # read_stream: client reads server responses from here
    # write_stream: client sends requests to server here
    async with stdio_client(server_params) as (read_stream, write_stream):

        # Wrap raw streams in an MCP session object.
        async with ClientSession(read_stream, write_stream) as session:

            # ── Handshake / Session Initialization ──────────────
            # This exchanges capabilities and prepares the session.
            await session.initialize()
            print("✅ MCP session initialized!\n")

            # ── STEP 1: Discover Tools ─────────────────────────
            # Ask server: "What callable tools do you expose?"
            tools_result = await session.list_tools()
            print(f"🔧 Server exposes {len(tools_result.tools)} tool(s):")

            for tool in tools_result.tools:
                # Some servers may not provide description text.
                desc = tool.description or "(no description provided)"
                print(f"   • {tool.name}: {desc[:80]}...")

                # inputSchema is JSON Schema the model/client can use
                # for validation + argument construction.
                schema_preview = json.dumps(tool.inputSchema, indent=6)[:200]
                print(f"     Input Schema: {schema_preview}...")
            print()

            # ── STEP 2: Discover Resources ─────────────────────
            # Ask server: "What readable resources (URIs) exist?"
            resources_result = await session.list_resources()
            print(f"📚 Server exposes {len(resources_result.resources)} resource(s):")

            for resource in resources_result.resources:
                res_desc = resource.description or "(no description provided)"
                print(f"   • {resource.uri}: {res_desc[:80]}...")
            print()

            # ── STEP 3: Call a Tool ────────────────────────────
            # Invoke a discovered tool by name with JSON arguments.
            print("🌤️  Calling get_weather for Tokyo (35.6762, 139.6503)...")
            result = await session.call_tool(
                "get_weather",
                arguments={"latitude": 35.6762, "longitude": 139.6503},
            )

            # MCP tool results return a content list; for this demo, we
            # expect the first content block to be text.
            print(f"   Result: {result.content[0].text}")
            print()

            # ── STEP 4: Read a Resource ────────────────────────
            # Resource URIs are read-only data endpoints.
            print("📖 Reading weather://codes resource...")
            resource = await session.read_resource("weather://codes")

            # Parse the JSON text payload into a Python dict.
            codes = json.loads(resource.contents[0].text)
            print(f"   Found {len(codes)} weather codes. First few:")

            for code_num, desc in list(codes.items())[:5]:
                print(f"     Code {code_num}: {desc}")
            print()

            print("🎉 MCP Client demo complete!")


# In notebooks, asyncio.run(...) works here because nest_asyncio was applied earlier.
asyncio.run(demo_mcp_client())

🔌 Connecting to weather_server.py via stdio...

✅ MCP session initialized!

🔧 Server exposes 2 tool(s):
   • get_weather: Get the current weather for a location.

    Args:
        latitude: Latitude of...
     Input Schema: {
      "properties": {
            "latitude": {
                  "title": "Latitude",
                  "type": "number"
            },
            "longitude": {
                  "title": "Longit...
   • get_weather_comparison: Compare current weather between two cities.

    Args:
        city1_lat: Latitu...
     Input Schema: {
      "properties": {
            "city1_lat": {
                  "title": "City1 Lat",
                  "type": "number"
            },
            "city1_lon": {
                  "title": "City...

📚 Server exposes 1 resource(s):
   • weather://codes: A lookup table of WMO weather interpretation codes....

🌤️  Calling get_weather for Tokyo (35.6762, 139.6503)...
   Result: Error executing tool get_weather: [SSL: CERTIFICATE_VERI

### 💡 What Just Happened?

We connected to our MCP server and:

1. **`list_tools()`** — The client *discovered* what tools the server offers, including their names, descriptions, and input schemas. **We never hard-coded this!** The server generated it all from the decorated Python functions.

2. **`list_resources()`** — Same for data resources.

3. **`call_tool()`** — We called the `get_weather` tool by name with typed arguments. The MCP protocol handled serialization, transport, execution on the server side, and deserialization of the result.

4. **`read_resource()`** — We read the weather codes lookup table.

**This is the magic of MCP:** the client is *generic*. It works with ANY MCP server — weather, database, GitHub, Slack — without any changes to the client code. Just point it at a different server.

---

# Part 6 · The OpenAI Agents SDK — MCP Made Even Easier

The OpenAI Agents SDK has **built-in MCP support**. This means you don't even need to write the tool-discovery-and-routing loop yourself — the SDK handles it all.

Let's see how clean this gets.

In [35]:
# ══════════════════════════════════════════════════════════════
# OpenAI Agents SDK + MCP — The Cleanest Approach
# ══════════════════════════════════════════════════════════════

from agents import Agent, Runner, enable_verbose_stdout_logging
from agents.mcp import MCPServerStdio
#enable_verbose_stdout_logging()

async def agents_sdk_demo():
    """
    Using OpenAI Agents SDK with our MCP server.
    Notice how we don't write ANY tool routing code!
    """

    mcp_server = MCPServerStdio(
        params={
            "command": "python",
            "args": ["weather_server.py"]
        }
    )

    async with mcp_server:
        agent = Agent(
            name="Weather Assistant",
            instructions=(
                "You are a helpful weather assistant. "
                "Use the available tools to answer questions about weather. "
                "Always provide temperatures in both Celsius and Fahrenheit."
            ),
            mcp_servers=[mcp_server],
        )

        print("🤖 Running agent with query...\n")
        result = await Runner.run(
            starting_agent=agent,
            input="What's the weather in New York City right now? "
                  "Also, what do the weather codes mean?"
        )

        print("💬 Agent's Response:")
        print(result.final_output)


asyncio.run(agents_sdk_demo())

🤖 Running agent with query...

💬 Agent's Response:
I couldn't retrieve the current weather for New York City due to a technical issue. However, I can explain what weather codes mean:

Weather codes are numerical values used by weather services to represent specific weather conditions for easy reference and automation. Some common weather codes include:
- 0: Clear sky
- 1-3: Mainly clear to partly cloudy
- 45: Fog
- 61-67: Rain (light to moderate, sometimes mixed with snow)
- 80-82: Showers
- 95-99: Thunderstorms

If you’d like, I can try to get the weather again or provide a more detailed list of weather codes. Let me know!


Runner.run(...) is the orchestration engine. The SDK docs describe the run loop like this:

call the LLM for the current agent and current input

inspect the output

if the model returns final output, stop

if the model returns tool calls, run those tool calls, append the results, and run the model again

repeat until final output or a turn limit is hit

this single line is hiding a lot of machinery.

### 🎯 Count the Lines

With the OpenAI Agents SDK + MCP, our entire application is:
- **3 lines** to set up the MCP server connection  
- **5 lines** to create the agent  
- **3 lines** to run a query  

That's **~11 lines** vs. the **~90 lines** we wrote in Part 1. And this version is:
- More robust (protocol-level error handling)  
- More flexible (works with any MCP server)  
- More maintainable (adding tools = 0 client changes)  

## For a Multi Turn Conversation. i.e. User, Assistant, then User, Assistant...

In [36]:
import asyncio
from agents import Agent, Runner, SQLiteSession
from agents.mcp import MCPServerStdio

async def chatbot():
    mcp_server = MCPServerStdio(
        params={
            "command": "python",
            "args": ["weather_server.py"],
        },
        cache_tools_list=True,
    )

    session = SQLiteSession("weather_chat_1")

    async with mcp_server:
        agent = Agent(
            name="Weather Assistant",
            instructions=(
                "You are a helpful weather assistant. "
                "Use the available tools to answer questions about weather. "
                "Always provide temperatures in both Celsius and Fahrenheit."
            ),
            mcp_servers=[mcp_server],
        )

        print("Chat started. Type 'exit' to stop.\n")

        while True:
            user_message = input("You: ").strip()
            if user_message.lower() in {"exit", "quit"}:
                break
            if not user_message:
                continue

            result = await Runner.run(
                starting_agent=agent,
                input=user_message,
                session=session,
                max_turns=8,
            )

            print(f"Assistant: {result.final_output}\n")

asyncio.run(chatbot())

Chat started. Type 'exit' to stop.



---

# Part 7 · Real-World MCP — A Practical Notes Server

Let's go beyond weather and build a more practical MCP server: a **note-taking system** that the LLM can use to store and retrieve information. This shows how MCP servers can manage state and perform CRUD operations.

In [37]:
# ══════════════════════════════════════════════════════════════
# A More Practical MCP Server: Notes Manager
# ══════════════════════════════════════════════════════════════

notes_server_code = '''
import json
from datetime import datetime
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("notes_mcp")
NOTES: dict[str, dict] = {}


@mcp.tool()
async def create_note(title: str, content: str, tags: str = "") -> str:
    """Create a new note with a title, content, and optional comma-separated tags.

    Args:
        title: The title of the note
        content: The main content/body of the note
        tags: Optional comma-separated tags (e.g., "work,urgent,meeting")
    """
    note_id = f"note_{len(NOTES) + 1}"
    NOTES[note_id] = {
        "title": title, "content": content,
        "tags": [t.strip() for t in tags.split(",") if t.strip()] if tags else [],
        "created_at": datetime.now().isoformat(),
    }
    return json.dumps({"status": "created", "note_id": note_id, "title": title})


@mcp.tool()
async def search_notes(query: str) -> str:
    """Search notes by title, content, or tags.

    Args:
        query: Search term to find in note titles, content, and tags
    """
    query_lower = query.lower()
    matches = []
    for note_id, note in NOTES.items():
        if (query_lower in note["title"].lower() or
            query_lower in note["content"].lower() or
            any(query_lower in tag.lower() for tag in note["tags"])):
            matches.append({"id": note_id, **note})
    if not matches:
        return json.dumps({"message": "No notes found.", "count": 0})
    return json.dumps({"notes": matches, "count": len(matches)}, indent=2, default=str)


@mcp.tool()
async def list_all_notes() -> str:
    """List all saved notes with their titles and IDs."""
    if not NOTES:
        return json.dumps({"message": "No notes yet!", "count": 0})
    summaries = [{"id": nid, "title": n["title"], "tags": n["tags"],
                  "preview": n["content"][:100]} for nid, n in NOTES.items()]
    return json.dumps({"notes": summaries, "count": len(summaries)}, indent=2, default=str)


@mcp.tool()
async def delete_note(note_id: str) -> str:
    """Delete a note by its ID.

    Args:
        note_id: The ID of the note to delete (e.g., note_1)
    """
    if note_id not in NOTES:
        return json.dumps({"error": f"Note {note_id} not found."})
    title = NOTES.pop(note_id)["title"]
    return json.dumps({"status": "deleted", "note_id": note_id, "title": title})


@mcp.resource("notes://all")
def all_notes_resource() -> str:
    """Browse all notes currently stored in the system."""
    return json.dumps(NOTES, indent=2, default=str)


if __name__ == "__main__":
    mcp.run()
'''

with open("notes_server.py", "w") as f:
    f.write(notes_server_code)

print("✅ notes_server.py written!")
print("  🔧 Tools: create_note, search_notes, list_all_notes, delete_note")
print("  📚 Resources: notes://all")

✅ notes_server.py written!
  🔧 Tools: create_note, search_notes, list_all_notes, delete_note
  📚 Resources: notes://all


In [38]:
# ══════════════════════════════════════════════════════════════
# Use the Notes Server with OpenAI Agents SDK
# ══════════════════════════════════════════════════════════════

async def notes_agent_demo():
    mcp_server = MCPServerStdio(
        params={"command": "python", "args": ["notes_server.py"]}
    )

    async with mcp_server:
        agent = Agent(
            name="Note-Taking Assistant",
            instructions=(
                "You are a personal note-taking assistant. "
                "Help the user create, search, list, and manage their notes. "
                "When creating notes, suggest relevant tags. "
                "Always confirm what you've done after each action."
            ),
            mcp_servers=[mcp_server],
        )

        queries = [
            "Create a note titled 'MCP Learning' with content: 'MCP is a protocol that standardizes "
            "how LLMs connect to tools. Key concepts: servers, clients, tools, resources, transports.' "
            "Tags: learning, mcp, ai",
            "Create another note titled 'Grocery List' with content: 'Milk, eggs, bread, avocados, "
            "coffee beans' Tags: personal, shopping",
            "Now search for anything related to 'protocol'",
            "List all my notes",
        ]

        for i, query in enumerate(queries, 1):
            print(f"\n{'='*60}")
            print(f"📝 Query {i}: {query[:80]}...")
            print(f"{'='*60}")
            result = await Runner.run(starting_agent=agent, input=query)
            print(f"\n🤖 Agent: {result.final_output}")

asyncio.run(notes_agent_demo())


📝 Query 1: Create a note titled 'MCP Learning' with content: 'MCP is a protocol that standa...

🤖 Agent: Your note titled "MCP Learning" has been created with the content you provided and the tags "learning," "mcp," and "ai."

Let me know if you'd like to add, edit, or search for more notes!

📝 Query 2: Create another note titled 'Grocery List' with content: 'Milk, eggs, bread, avoc...

🤖 Agent: Your note titled "Grocery List" with the content "Milk, eggs, bread, avocados, coffee beans" has been created and tagged with "personal, shopping".

Would you like to view, edit, or add more notes?

📝 Query 3: Now search for anything related to 'protocol'...

🤖 Agent: I found one note related to "protocol":

- Title: MCP Learning
  - Content: MCP is a protocol that standardizes how LLMs connect to tools. Key concepts: servers, clients, tools, resources, transports.
  - Tags: learning, mcp, ai

Would you like to view more details, edit, or take any action on this note?

📝 Query 4: List all my n

---

# Part 8 · The True Power — Connecting Multiple MCP Servers

The real magic of MCP shines when you connect an agent to **multiple servers simultaneously**. Each server is a specialist — and the agent orchestrates them all.

Let's connect our weather server AND notes server to the same agent:

In [40]:
# ══════════════════════════════════════════════════════════════
# Multi-Server Agent — Weather + Notes
# ══════════════════════════════════════════════════════════════

async def multi_server_demo():
    weather_server = MCPServerStdio(
        params={"command": "python", "args": ["weather_server.py"]}
    )
    notes_server = MCPServerStdio(
        params={"command": "python", "args": ["notes_server.py"]}
    )

    async with weather_server, notes_server:
        agent = Agent(
            name="Personal Assistant",
            instructions=(
                "You are a personal assistant with access to weather data and a notes system. "
                "You can check weather anywhere in the world and manage notes. "
                "When the user asks about weather, check it and offer to save the info as a note."
            ),
            mcp_servers=[weather_server, notes_server],
        )

        print("🚀 Agent has tools from BOTH servers!\n")

        result = await Runner.run(
            starting_agent=agent,
            input=(
                "Check the weather in Tokyo and Paris right now, "
                "then save a note with the comparison results. "
                "Title it 'Weather Check' with tags: weather, travel"
            )
        )

        print("💬 Agent's Response:")
        print(result.final_output)

asyncio.run(multi_server_demo())

🚀 Agent has tools from BOTH servers!

💬 Agent's Response:
The current weather in Tokyo is 12.7°C with a wind speed of 5.7 km/h, while Paris is at 7.4°C with a wind speed of 4.7 km/h. Tokyo is currently both warmer and windier than Paris.

I've saved a note with this comparison titled "Weather Check" and tagged it with "weather" and "travel." If you need to reference or update this note later, just let me know!


## Alternative

```python

servers = [weather_server, notes_server, calendar_server]

    # Recommended when you have multiple MCP servers
    async with MCPServerManager(servers) as manager:
        agent = Agent(
            name="Multi-Server Assistant",
            instructions=(
                "You are a helpful assistant.\n"
                "Use the weather server for weather questions.\n"
                "Use the notes server for note/document lookup.\n"
                "Use the calendar server for schedule-related questions.\n"
                "If multiple servers are useful, use them."
            ),
            mcp_servers=manager.active_servers,
        )

        result = await Runner.run(
            starting_agent=agent,
            input=(
                "What's the weather in New York right now, "
                "and do I have any meetings today related to weather ops?"
            ),
            max_turns=8,
        )

        print(result.final_output)

asyncio.run(multi_server_demo())
```

### 🔥 Why This Is Revolutionary

We just gave our agent **two entirely different capabilities** (weather + notes) by simply listing two MCP servers. We wrote:
- Zero routing code  
- Zero schema definitions  
- Zero integration glue  

To add a third capability (say, email), we'd just add a third MCP server to the list. The agent automatically discovers and uses all tools across all servers.

**This is the N + M promise of MCP in action.** Add tools by adding servers, not by rewriting your application.

---

# Part 9 · Transports — How Clients and Servers Actually Communicate

MCP is **transport-agnostic**. The protocol logic (tool calls, results, etc.) sits on top of whatever communication layer you choose.

## The Two Main Transports

### 1. stdio (Standard Input/Output)

```
┌──────────────┐      stdin/stdout      ┌──────────────┐
│  MCP Client  │ ◄──────────────────► │  MCP Server  │
│  (your app)  │   (subprocess pipe)    │ (child proc) │
└──────────────┘                        └──────────────┘
```

- **How:** The client launches the server as a child process and communicates via stdin/stdout pipes.  
- **When:** Local development, CLI tools, desktop apps (like Claude Desktop).  
- **Pros:** Simple, fast, no networking needed, great for local tools.  
- **Cons:** Server must be on the same machine.

### 2. Streamable HTTP

```
┌──────────────┐      HTTP POST/GET      ┌──────────────┐
│  MCP Client  │ ◄───────────────────► │  MCP Server  │
│  (your app)  │   (network request)     │ (remote host)│
└──────────────┘                         └──────────────┘
```

- **How:** Client sends HTTP requests to the server's MCP endpoint.  
- **When:** Remote servers, cloud deployments, production services.  
- **Pros:** Works across networks, scalable, supports multiple clients.  
- **Cons:** More complex setup, needs HTTPS in production.

### 3. SSE — Server-Sent Events (Legacy)
- Still supported for backward compatibility.  
- New projects should use Streamable HTTP.

---

## 📝 Configuration in Practice

### For Claude Desktop (JSON config file):

```json
{
  "mcpServers": {
    "weather": {
      "command": "python",
      "args": ["/path/to/weather_server.py"]
    },
    "github": {
      "command": "npx",
      "args": ["-y", "@modelcontextprotocol/server-github"],
      "env": { "GITHUB_TOKEN": "ghp_your_token_here" }
    }
  }
}
```

### For OpenAI Agents SDK (Python):

```python
# stdio (local)
server = MCPServerStdio(params={
    "command": "python",
    "args": ["weather_server.py"]
})

# Streamable HTTP (remote)
from agents.mcp import MCPServerStreamableHttp
server = MCPServerStreamableHttp(params={
    "url": "https://my-mcp-server.example.com/mcp"
})

# SSE (legacy remote)
from agents.mcp import MCPServerSse
server = MCPServerSse(params={
    "url": "http://localhost:8000/sse"
})
```

---

# Part 10 · Recap — Everything We Learned

## 📍 Where We Started
We tried to connect an LLM to a weather API manually using the Responses API. It took ~90 lines of boilerplate — tool schemas, function routing, argument parsing — for **just one tool.** Scaling to multiple tools would multiply this pain.

## 🔌 What We Discovered
**MCP (Model Context Protocol)** — an open standard that eliminates the glue code by defining a universal way for LLMs to discover and use tools.

## 🏗️ What We Built

| Component | What It Does | Key Code |
|-----------|-------------|----------|
| **MCP Server** (weather) | Exposes weather tools + resources | `@mcp.tool()` decorator on async functions |
| **MCP Server** (notes) | Full CRUD note-taking system | 4 tools + 1 resource, ~80 lines |
| **MCP Client** | Connects, discovers, and calls tools | `session.list_tools()`, `session.call_tool()` |
| **LLM + MCP bridge** | GPT-4o-mini using MCP tools via Responses API | Auto-converts MCP schemas → OpenAI format |
| **OpenAI Agents SDK** | Cleanest integration — ~11 lines | `Agent(mcp_servers=[...])` |
| **Multi-server agent** | Weather + Notes in one agent | Just list both servers! |

## 🗝️ Key Concepts

```
┌─────────────────────────────────────────────────────────────┐
│                     MCP MENTAL MODEL                         │
│                                                              │
│   HOST (Your App)                                            │
│     └── CLIENT ─── [transport: stdio / HTTP] ──→ SERVER     │
│                                                  ├── Tools   │
│                                                  ├── Resources│
│                                                  └── Prompts │
│                                                              │
│   • Server defines capabilities                              │
│   • Client discovers them at runtime                         │
│   • Protocol handles serialization, routing, errors          │
│   • Transport is swappable (local stdio ↔ remote HTTP)       │
│   • N servers + M apps = N+M integrations (not N×M)         │
└─────────────────────────────────────────────────────────────┘
```

## 🌍 The MCP Ecosystem

| Server | What It Does |
|--------|-------------|
| `@modelcontextprotocol/server-filesystem` | Read/write local files |
| `@modelcontextprotocol/server-github` | GitHub operations (issues, PRs, repos) |
| `@modelcontextprotocol/server-slack` | Slack messaging |
| `@modelcontextprotocol/server-fetch` | Fetch web pages |
| `@modelcontextprotocol/server-postgres` | PostgreSQL queries |
| Community servers | Jira, Notion, Google Drive, S3, and hundreds more |

To use any of them, you just configure the connection — your client code stays the same.

## 🧪 Challenge Exercises

Ready to solidify your understanding? Try these:

1. **Add a Tool:** Add a `get_forecast` tool to `weather_server.py` that returns a 7-day forecast using the Open-Meteo API's `daily` parameter. Verify the agent discovers it automatically.

2. **Build a New Server:** Create a `calculator_server.py` that exposes tools for `add`, `subtract`, `multiply`, `divide`, and `evaluate_expression`. Connect it to an agent.

3. **Multi-Server Mashup:** Connect your calculator + weather + notes servers to a single agent. Ask: *"What's the temperature difference between Tokyo and Paris? Save the result in a note."*

4. **Remote Transport:** Modify your weather server to use Streamable HTTP (`mcp.run(transport="streamable_http", port=8000)`) and connect to it via `MCPServerStreamableHttp`.

5. **Explore the Registry:** Visit [github.com/modelcontextprotocol/servers](https://github.com/modelcontextprotocol/servers) and try connecting a community MCP server to your agent.

---

## 📚 Further Reading

- **MCP Official Docs:** [modelcontextprotocol.io](https://modelcontextprotocol.io)  
- **MCP Python SDK:** [github.com/modelcontextprotocol/python-sdk](https://github.com/modelcontextprotocol/python-sdk)  
- **OpenAI Agents SDK (MCP):** [openai.github.io/openai-agents-python/mcp/](https://openai.github.io/openai-agents-python/mcp/)  
- **OpenAI Responses API:** [platform.openai.com/docs/api-reference/responses](https://platform.openai.com/docs/api-reference/responses)  
- **MCP Server Registry:** [github.com/modelcontextprotocol/servers](https://github.com/modelcontextprotocol/servers)  

---
